# Saklar Uji-Asap SMOKE TEST


In [1]:
import os
os.environ["SMOKE"] = "1"
os.environ["ZIP_PATH"] = "/content/dataset_v2_frozen.zip"

# 1 Setup + verifikasi hash + larang lockbox

In [2]:
import os, sys, re, json, time, random, hashlib, zipfile, subprocess, copy
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=False)
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

BASE = os.environ.get("DATA_DIR", "/content/data_colab")
ZIP_PATH = os.environ.get("ZIP_PATH", "/content/drive/MyDrive/dataset_v2_frozen.zip")
if not os.path.exists(os.path.join(BASE, "train.csv")):
    if not os.path.exists(ZIP_PATH):
        from google.colab import drive
        drive.mount("/content/drive")
    zipfile.ZipFile(ZIP_PATH).extractall(BASE)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""): h.update(ch)
    return h.hexdigest().lower()

man = pd.read_csv(f"{BASE}/MANIFEST_sha256.csv").set_index("berkas").sha256.str.lower()
FILES = ["train.csv", "val.csv", "dev_test.csv", "eval_benar.csv", "eval_kompas_cekfakta.csv", "text_pipeline.py"]
for f in FILES:
    assert os.path.exists(f"{BASE}/{f}") and sha(f"{BASE}/{f}") == man.get(f), f"hash tidak cocok: {f}"
assert not any("lockbox" in x.lower() for x in os.listdir(BASE)), "LOCKBOX tidak boleh ada di Colab"
sys.path.insert(0, BASE)
import text_pipeline as tp
assert tp.PIPELINE_VERSION == "tp-1.6"
print("Data & pipeline terverifikasi.")

Data & pipeline terverifikasi.


# 2 Konfigurasi TERKUNCI

In [3]:
CFG = dict(
    MODEL_NAME="indobenchmark/indobert-base-p1", MAX_LEN=128, BATCH=32, EPOCHS=4, LR=2e-5, WD=0.01,
    WARMUP=0.1, DROPOUT=0.3, FREEZE_LAYERS=8, SEEDS=[42, 123, 999], LOSSES=["ce", "focal"], FOCAL_GAMMA=2.0,
)
SMOKE = os.environ.get("SMOKE") == "1"          # uji-asap: 1 seed, 1 epoch, CE. Hasilnya TIDAK dilaporkan.
if SMOKE:
    CFG.update(EPOCHS=1, SEEDS=[42], LOSSES=["ce"]); print(">> MODE SMOKE: hanya untuk menangkap error; jangan dicatat sebagai hasil")
PFX = "smoke_" if SMOKE else ""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

tr, va, te = [pd.read_csv(f"{BASE}/{s}.csv") for s in ("train", "val", "dev_test")]
for d in (tr, va, te):
    d["text"] = d.text.fillna("").astype(str); d["title"] = d.title.fillna("").astype(str)
print({"train": len(tr), "val": len(va), "dev_test": len(te)})
tok = AutoTokenizer.from_pretrained(CFG["MODEL_NAME"])

# Panjang token dihitung dari TRAIN saja; MAX_LEN naik ke 160 bila >1% terpotong
lens = np.array([len(x) for x in tok(tr.text.tolist(), add_special_tokens=True, truncation=False)["input_ids"]])
share = float((lens > CFG["MAX_LEN"]).mean())
print(f"Token train: p50={np.percentile(lens,50):.0f} p95={np.percentile(lens,95):.0f} p99={np.percentile(lens,99):.0f} maks={lens.max()} | terpotong@{CFG['MAX_LEN']}: {share*100:.2f}%")
if share > 0.01:
    CFG["MAX_LEN"] = 160; print(">> MAX_LEN dinaikkan ke 160 (aturan: >1% terpotong)")

def encode(texts):
    e = tok(list(texts), max_length=CFG["MAX_LEN"], padding="max_length", truncation=True, return_tensors="pt")
    return e["input_ids"], e["attention_mask"]

def loader(df, shuffle, seed=0):
    ids, am = encode(df.text.values)
    ds = TensorDataset(ids, am, torch.tensor(df.label.values, dtype=torch.long))
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=CFG["BATCH"], shuffle=shuffle, generator=g if shuffle else None)

val_loader, dev_loader = loader(va, False), loader(te, False)

>> MODE SMOKE: hanya untuk menangkap error; jangan dicatat sebagai hasil
Device: cpu 
{'train': 1660, 'val': 236, 'dev_test': 236}


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Token train: p50=68 p95=91 p99=105 maks=141 | terpotong@128: 0.12%


# 3 Model & loss

In [4]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0): super().__init__(); self.gamma = gamma
    def forward(self, logits, y):
        ce = F.cross_entropy(logits, y, reduction="none"); pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

class Clf(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(CFG["MODEL_NAME"])
        for p in self.bert.embeddings.parameters(): p.requires_grad = False
        for i in range(CFG["FREEZE_LAYERS"]):
            for p in self.bert.encoder.layer[i].parameters(): p.requires_grad = False
        self.drop = nn.Dropout(CFG["DROPOUT"]); self.head = nn.Linear(self.bert.config.hidden_size, 2)
    def forward(self, ids, am):
        return self.head(self.drop(self.bert(input_ids=ids, attention_mask=am).last_hidden_state[:, 0, :]))

@torch.no_grad()
def predict(model, dl):
    model.eval(); P, Pr = [], []
    for ids, am, _ in dl:
        lg = model(ids.to(device), am.to(device))
        P.append(lg.argmax(1).cpu().numpy()); Pr.append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
    return np.concatenate(P), np.concatenate(Pr)

# 4 Eval set di luar distribusi judul saja, praproses sama dengan pipeline

In [5]:
nk = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
seen = set(tr.title.map(nk)) | set(va.title.map(nk)) | set(te.title.map(nk))
prep_title = lambda t: tp.ensure_end(tp.norm_title(str(t)))

def load_eval(path, filter_klaim=False):
    d = pd.read_csv(path); d["title"] = d.title.fillna("").astype(str)
    if filter_klaim:
        tag = d.title.str.match(r"(?i)^\s*\[(?:hoaks|hoax|salah|disinformasi)\]")
        if tag.sum() > 0: d = d[tag]
    d["k"] = d.title.map(nk); d = d.drop_duplicates("k"); d = d[~d.k.isin(seen)]
    d = d.assign(text=d.title.map(prep_title)).assign(label=0)   # label dummy; yang dilihat adalah proporsi prediksi hoaks
    return d.reset_index(drop=True)

EV = {"eval_benar": load_eval(f"{BASE}/eval_benar.csv"),
      "eval_kompas": load_eval(f"{BASE}/eval_kompas_cekfakta.csv", filter_klaim=True)}
EV_LOADERS = {k: loader(v, False) for k, v in EV.items()}
print({k: len(v) for k, v in EV.items()})

{'eval_benar': 432, 'eval_kompas': 20}


# 5 Satu eksperimen satu seed, satu loss

In [6]:
def train_one(loss_type, seed):
    set_seed(seed)
    model = Clf().to(device)
    crit = FocalLoss(CFG["FOCAL_GAMMA"]) if loss_type == "focal" else nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = AdamW(params, lr=CFG["LR"], weight_decay=CFG["WD"])
    tl = loader(tr, True, seed)
    total = len(tl) * CFG["EPOCHS"]
    sch = get_linear_schedule_with_warmup(opt, int(total * CFG["WARMUP"]), total)
    best_f1, best_state, best_ep, hist = -1.0, None, 0, []
    for ep in range(1, CFG["EPOCHS"] + 1):
        model.train(); tot = 0.0
        for ids, am, y in tl:
            opt.zero_grad()
            loss = crit(model(ids.to(device), am.to(device)), y.to(device))
            loss.backward(); nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sch.step(); tot += loss.item()
        pv, _ = predict(model, val_loader)
        f1 = f1_score(va.label, pv, average="macro"); acc = accuracy_score(va.label, pv)
        hist.append(dict(epoch=ep, train_loss=tot / len(tl), val_acc=acc, val_f1=f1))
        print(f"  [{loss_type} s{seed}] ep{ep} train_loss={tot/len(tl):.4f} val_acc={acc*100:.2f} val_f1={f1*100:.2f}")
        if f1 > best_f1:                       # pemilihan HANYA dari val
            best_f1, best_ep = f1, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    pv, prv = predict(model, val_loader); pd_, prd = predict(model, dev_loader)
    out = dict(loss=loss_type, seed=seed, best_epoch=best_ep,
               val_acc=accuracy_score(va.label, pv), val_f1=f1_score(va.label, pv, average="macro"),
               dev_acc=accuracy_score(te.label, pd_), dev_f1=f1_score(te.label, pd_, average="macro"))
    for k, dl in EV_LOADERS.items():
        p, _ = predict(model, dl); out[f"{k}_share_hoaks"] = float(p.mean())
    preds = dict(val=(pv, prv), dev=(pd_, prd))
    del model, opt; torch.cuda.empty_cache()
    return out, hist, preds

# 6 Jalankan SEMUA konfigurasi x seed urutan tetap, tanpa memilih seed terbaik

In [7]:
RUNS, HIST, PRED = [], {}, {}
t0 = time.time()
for loss_type in CFG["LOSSES"]:
    for seed in CFG["SEEDS"]:
        print(f"\n=== {loss_type.upper()} | seed {seed} ===")
        r, h, p = train_one(loss_type, seed)
        RUNS.append(r); HIST[f"{loss_type}_{seed}"] = h; PRED[f"{loss_type}_{seed}"] = p
print(f"\nTotal waktu: {(time.time()-t0)/60:.1f} menit")


=== CE | seed 42 ===


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [ce s42] ep1 train_loss=0.3537 val_acc=93.64 val_f1=93.63

Total waktu: 33.5 menit


# Ringkasan + aturan pemilihan konfigurasi

In [8]:
R = pd.DataFrame(RUNS)
pd.set_option("display.width", 200)
print(R.round(4).to_string(index=False))
agg = R.groupby("loss")[["val_acc", "val_f1", "dev_acc", "dev_f1", "eval_benar_share_hoaks", "eval_kompas_share_hoaks"]].agg(["mean", "std"]).round(4)
print("\nRerata ± sd lintas seed:"); print(agg.to_string())
mv = R.groupby("loss").val_f1.mean()
pilih = "ce" if abs(mv.get("ce", 0) - mv.get("focal", 0)) < 0.01 else mv.idxmax()
print(f"\nAturan pemilihan (rerata val macro-F1; selisih <1 poin -> CE): konfigurasi terpilih = {pilih.upper()}")
print("dev_test hanya dilaporkan, tidak dipakai memilih. Selisih <±4 poin pada n≈236 tidak bermakna.")

loss  seed  best_epoch  val_acc  val_f1  dev_acc  dev_f1  eval_benar_share_hoaks  eval_kompas_share_hoaks
  ce    42           1   0.9364  0.9363    0.911   0.911                   0.669                     0.35

Rerata ± sd lintas seed:
     val_acc      val_f1     dev_acc     dev_f1     eval_benar_share_hoaks     eval_kompas_share_hoaks    
        mean std    mean std    mean std   mean std                   mean std                    mean std
loss                                                                                                      
ce    0.9364 NaN  0.9363 NaN   0.911 NaN  0.911 NaN                  0.669 NaN                    0.35 NaN

Aturan pemilihan (rerata val macro-F1; selisih <1 poin -> CE): konfigurasi terpilih = CE
dev_test hanya dilaporkan, tidak dipakai memilih. Selisih <±4 poin pada n≈236 tidak bermakna.


# 8 Simpan log lengkap catat di protokol

In [9]:
LOG = dict(config=CFG, pipeline=tp.PIPELINE_VERSION,
           data_hash={f: man.get(f) for f in FILES}, runs=RUNS, history=HIST, terpilih=pilih,
           torch=torch.__version__, transformers=__import__('transformers').__version__, device=str(device))
json.dump(LOG, open(f"{PFX}log_training.json", "w"), indent=2, default=float)
rows = pd.DataFrame({"split": ["val"] * len(va) + ["dev_test"] * len(te),
                     "id": list(va.id) + list(te.id), "label": list(va.label) + list(te.label)})
for k, p in PRED.items():
    rows[f"pred_{k}"] = np.concatenate([p["val"][0], p["dev"][0]])
    rows[f"prob_{k}"] = np.concatenate([p["val"][1], p["dev"][1]])
rows.to_csv(f"{PFX}prediksi_val_dev.csv", index=False)     # untuk uji berpasangan (McNemar) & analisis error
print(f"Tersimpan: {PFX}log_training.json, {PFX}prediksi_val_dev.csv")
try:                                    # sesi Colab sementara: unduh hasil ke laptop
    from google.colab import files
    for f in (f"{PFX}log_training.json", f"{PFX}prediksi_val_dev.csv"): files.download(f)
except Exception as e:
    print("Unduh manual dari panel Files:", e)

Tersimpan: smoke_log_training.json, smoke_prediksi_val_dev.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 9 Perbandingan berpasangan IndoBERT vs baseline TF-IDF McNemar eksak

In [11]:
from scipy.stats import binomtest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def tfidf_preds(Xa, ya, Xb):
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    c = LogisticRegression(max_iter=2000, random_state=0).fit(v.fit_transform(Xa), ya)
    return c.predict(v.transform(Xb))

def mcnemar(y, a, b):
    ra, rb = (a == y), (b == y)
    n_a, n_b = int((ra & ~rb).sum()), int((~ra & rb).sum())     # a benar b salah | a salah b benar
    n = n_a + n_b
    return n_a, n_b, (binomtest(min(n_a, n_b), n, 0.5).pvalue if n > 0 else 1.0)

base = {"val": tfidf_preds(tr.text, tr.label, va.text), "dev": tfidf_preds(tr.text, tr.label, te.text)}
ys = {"val": va.label.to_numpy(), "dev": te.label.to_numpy()}
print("split | run | akurasi IndoBERT | akurasi baseline | IndoBERT-benar-saja | baseline-benar-saja | p (McNemar)")
for k, pr in PRED.items():
    for sp in ("val", "dev"):
        a = pr[sp][0]; n_a, n_b, pv = mcnemar(ys[sp], a, base[sp])
        print(f"{sp:<4} | {k:<10} | {(a==ys[sp]).mean()*100:5.1f}% | {(base[sp]==ys[sp]).mean()*100:5.1f}% | {n_a:>3} | {n_b:>3} | {pv:.3f}")
print("p besar (>0,05) = selisih tidak dapat dibedakan dari kebetulan pada n≈236.")

split | run | akurasi IndoBERT | akurasi baseline | IndoBERT-benar-saja | baseline-benar-saja | p (McNemar)
val  | ce_42      |  93.6% |  94.1% |   8 |   9 | 1.000
dev  | ce_42      |  91.1% |  91.5% |  14 |  15 | 1.000
p besar (>0,05) = selisih tidak dapat dibedakan dari kebetulan pada n≈236.
